In [20]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto_MLDM')

    GIT_BRANCH = 'refactor-continuo'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

Ambiente Colab rilevato. Inizializzazione in corso...
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 671 bytes | 335.00 KiB/s, done.
From https://github.com/SimoRinaldi/crop-spatial-classification
   56b4a28..610fd2e  refactor-continuo -> origin/refactor-continuo
branch 'refactor-continuo' set up to track 'origin/refactor-continuo'.
Reset branch 'refactor-continuo'
Your branch is up to date with 'origin/refactor-continuo'.
HEAD is now at 610fd2e fix 04 notebook
Setup ambiente Colab completato! Branch attivo su Colab:  refactor-continuo
✅ Collegamento ai dati riuscito! Cartella raw: /content/drive/MyDrive/Progetto_MLDM/data/raw


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

dataset_parquet = Path(f"{DATA_DIR}/processed/dataset/dataset.parquet")

dataset = pd.read_parquet(dataset_parquet)

y = dataset["Ground_Truth"].astype(int)

# Feature (Bande + Indici)
X = dataset[
    [
        "Blu_B02",
        "Verde_B03",
        "Rosso_B04",
        "NIR_B08",
        "SWIR1_B11",
        "SWIR2_B12",
        # "NDVI_01",
        # "NDVI_02",
        # "NDVI_03",
        # "NDVI_04",
        # "NDVI_05",
        # "NDVI_06",
        # "NDVI_07",
        # "NDVI_08",
        # "NDVI_09",
        # "NDVI_10",
        # "NDVI_11",
        # "NDVI_12",
        # "NDWI_01",
        # "NDWI_02",
        # "NDWI_03",
        # "NDWI_04",
        # "NDWI_05",
        # "NDWI_06",
        # "NDWI_07",
        # "NDWI_08",
        # "NDWI_09",
        # "NDWI_10",
        # "NDWI_11",
        # "NDWI_12",
        # "NDVI_max",
        # "NDVI_min",
        # "NDVI_amp",
        # "Peak_Month",
    ]
]

try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)

    print("🏆 Classification Report 🏆")
    print("-" * 50)
    print(classification_report(y_test, prediction, zero_division=0))
except Exception as e:
    print(f"Non ho abbastanza dati per fare il test. Errore: {e}")

🏆 Classification Report 🏆
--------------------------------------------------
              precision    recall  f1-score   support

        1110       0.34      0.32      0.33        50
        1120       0.36      0.37      0.37        43
        1130       0.56      0.67      0.61        42
        1140       0.76      0.89      0.82        35
        1150       0.54      0.36      0.43        42
        1210       0.26      0.21      0.23        42
        1220       0.12      0.14      0.13        37
        1310       0.41      0.26      0.32        50
        1410       0.33      0.53      0.41        51
        1420       0.70      0.85      0.77        33
        1430       0.33      0.38      0.36        52
        1440       0.65      0.64      0.65        56
        2100       0.46      0.57      0.50        46
        2200       0.49      0.31      0.38        59
        2310       0.21      0.23      0.22        39
        2320       0.35      0.43      0.38        44
    